In [1]:
import pandas as pd

df = pd.read_parquet(
    "../data/processed/orders_clean.parquet"
)

print(df.shape)

(81426, 31)


In [2]:
df["hubName"].value_counts()

hubName
Instant Foods(Noida)             18870
Instant Foods (SED)              10035
Crossline Events (Noida)          8834
Instant Foods (GZB)               8647
Rapid Enterprises                 7790
Cross Line Events (Ghaziabad)     7638
Cross Line Events (EDelhi)        7242
Crossline Events (Meerut)         7167
NB Enterprises (West Delhi)       5203
Name: count, dtype: int64

In [3]:
region_trends = (
    df.groupby(
        [
            "hubId",
            "hubName",
            "shopType",
            "skuNumber"
        ]
    )
    .agg(
        total_qty=(
            "effective_qty",
            "sum"
        ),

        unique_buyers=(
            "customerId",
            "nunique"
        ),

        total_orders=(
            "orderNumber",
            "nunique"
        )
    )
    .reset_index()
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders
0,HB002,Instant Foods(Noida),Chemist C,SKU00026,1,1,1
1,HB002,Instant Foods(Noida),Chemist C,SKU00030,1,1,1
2,HB002,Instant Foods(Noida),Chemist C,SKU00060,6,2,4
3,HB002,Instant Foods(Noida),Chemist C,SKU00062,4,1,2
4,HB002,Instant Foods(Noida),Chemist C,SKU00067,1,1,1


In [4]:
sku_lookup = (
    df[
        [
            "skuNumber",
            "itemName",
            "brand",
            "category"
        ]
    ]
    .drop_duplicates()
)

region_trends = (
    region_trends.merge(
        sku_lookup,
        on="skuNumber",
        how="left"
    )
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders,itemName,brand,category
0,HB002,Instant Foods(Noida),Chemist C,SKU00026,1,1,1,Pulse Litchi Candy Jar,DS Group,Confectionery
1,HB002,Instant Foods(Noida),Chemist C,SKU00030,1,1,1,Pulse Mix Candy Jar,DS Group,Confectionery
2,HB002,Instant Foods(Noida),Chemist C,SKU00060,6,2,4,Rajnigandha 4g,Rajnigandha,Pan Masala
3,HB002,Instant Foods(Noida),Chemist C,SKU00062,4,1,2,RG Pearl Elaichi MP Pouch 0.14g,DS Group,Mouth Fresheners
4,HB002,Instant Foods(Noida),Chemist C,SKU00067,1,1,1,RG Pearl Elaichi 0.14g Jar,DS Group,Mouth Fresheners


In [5]:
region_trends[
    "region_rank"
] = (
    region_trends
    .groupby(
        [
            "hubId",
            "shopType"
        ]
    )
    ["unique_buyers"]
    .rank(
        ascending=False,
        method="dense"
    )
)

region_trends.head()

,hubId,hubName,shopType,skuNumber,total_qty,unique_buyers,total_orders,itemName,brand,category,region_rank
0,HB002,Instant Foods(Noida),Chemist C,SKU00026,1,1,1,Pulse Litchi Candy Jar,DS Group,Confectionery,2.0
1,HB002,Instant Foods(Noida),Chemist C,SKU00030,1,1,1,Pulse Mix Candy Jar,DS Group,Confectionery,2.0
2,HB002,Instant Foods(Noida),Chemist C,SKU00060,6,2,4,Rajnigandha 4g,Rajnigandha,Pan Masala,1.0
3,HB002,Instant Foods(Noida),Chemist C,SKU00062,4,1,2,RG Pearl Elaichi MP Pouch 0.14g,DS Group,Mouth Fresheners,2.0
4,HB002,Instant Foods(Noida),Chemist C,SKU00067,1,1,1,RG Pearl Elaichi 0.14g Jar,DS Group,Mouth Fresheners,2.0


In [6]:
(
    region_trends[
        region_trends["hubName"]
        == "Instant Foods(Noida)"
    ]
    .sort_values(
        "region_rank"
    )
    [
        [
            "itemName",
            "unique_buyers",
            "region_rank"
        ]
    ]
    .head(10)
)

,itemName,unique_buyers,region_rank
429,Rajnigandha 4g,47,1.0
22,Rajnigandha 4g,74,1.0
809,Rajnigandha 4g,4,1.0
289,Rajnigandha 4g,136,1.0
694,Rajnigandha 4g,46,1.0
547,TZ 00 (with Silver) 0.45g,302,1.0
9,Center Fresh,2,1.0
150,TZ 00 (with Silver) 0.45g,81,1.0
2,Rajnigandha 4g,2,1.0
6,TZ 00 (with Silver) 0.45g,2,1.0


In [7]:
region_trends.to_parquet(
    "../data/processed/region_trends.parquet",
    index=False
)

print("Saved")

Saved


In [8]:
print(region_trends.shape)

print(
    region_trends["hubName"]
    .nunique()
)

print(
    region_trends["shopType"]
    .nunique()
)

(5197, 11)
9
10


In [9]:
region_trends.shape

(5197, 11)

In [10]:
region_trends.sort_values(
    "region_rank"
)[
    [
        "hubName",
        "shopType",
        "itemName",
        "region_rank"
    ]
].head(15)

,hubName,shopType,itemName,region_rank
22,Instant Foods(Noida),General A,Rajnigandha 4g,1.0
2203,Cross Line Events (Ghaziabad),General C,Rajnigandha 4g,1.0
2,Instant Foods(Noida),Chemist C,Rajnigandha 4g,1.0
9,Instant Foods(Noida),Chemist C,Center Fresh,1.0
2999,NB Enterprises (West Delhi),Chemist C,Britannia Tiger Krunch 60 Pcs,1.0
2998,NB Enterprises (West Delhi),Chemist A,Sunfeast Mom's Magic - Rich Butter,1.0
5168,Rapid Enterprises,Wholesaler,Rajnigandha 4g,1.0
5172,Rapid Enterprises,Wholesaler,TZ 00 (with Silver) 0.45g,1.0
2007,Crossline Events (Noida),Wholesaler,TZ 00 4.25g (6 Pcs),1.0
2008,Crossline Events (Noida),Wholesaler,TZ 00 (with Silver) 0.45g,1.0


In [11]:
region_trends.shape

(5197, 11)